In [1]:
import os
from pathlib import Path
from typing import List

from tqdm import tqdm
import numpy as np
import pandas as pd
try:
    import google.generativeai as genai
except ImportError as exc:
    raise ImportError("Install google-generativeai via `pip install google-generativeai`.") from exc


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
# ---- Configuration ----
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if GOOGLE_API_KEY is not None:
    GEMINI_API_KEY = GOOGLE_API_KEY
if not GEMINI_API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY in your environment before running this cell.")

genai.configure(api_key=GEMINI_API_KEY)

ROOT_DIR = Path("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2").expanduser()
if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

MODEL_NAME = "gemini-embedding-001"
# MODEL_NAME = "text-embedding-004"
BATCH_SIZE = 250  # number of .py files to embed before logging progress


In [27]:
def read_python_file(file_path: Path) -> str:
    """Return the contents of a Python file with a lightweight header."""
    return f"# File: {file_path.name}\n" + file_path.read_text(encoding="utf-8", errors="ignore")


def embed_text(texts: List[str]) -> List[float]:
    response = genai.embed_content(
        model=MODEL_NAME,
        content=texts,
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=768,
    )
    return response["embedding"]


def embed_python_file(file_path: Path) -> np.ndarray:
    source = read_python_file(file_path)
    if not source.strip():
        raise ValueError(f"{file_path} is empty or unreadable")
    return np.asarray(embed_text(source), dtype=np.float32)


In [28]:
df = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/data/datasets/labeled_data.csv')

In [29]:
records = []

python_files = df['code_summary'].values

total_files = len(python_files)
for batch_start in range(0, total_files, BATCH_SIZE):
    batch = python_files[batch_start : batch_start + BATCH_SIZE]
    print(
        f"Processing files {batch_start + 1}-{batch_start + len(batch)} / {total_files}"
    )

    embeddings = embed_text(batch)
    for summary, embedding_vector in zip(batch, embeddings):
        records.append(
            {
                "summary": summary,
                "embedding": embedding_vector,
            }
        )
 

Processing files 1-250 / 1679
Processing files 251-500 / 1679
Processing files 501-750 / 1679
Processing files 751-1000 / 1679
Processing files 1001-1250 / 1679
Processing files 1251-1500 / 1679
Processing files 1501-1679 / 1679


In [30]:
embedding_length = len(records[0]["embedding"])
rows = []
for record in records:
    row = {f"dim_{i+1}": value for i, value in enumerate(record["embedding"])}
    row["file"] = df.iloc[len(rows)]['file']
    rows.append(row)

embeddings_df = pd.DataFrame(rows)
print(
    f"Generated embeddings for {len(embeddings_df)} files with {embedding_length} dimensions."
)
display(embeddings_df.head())

OUTPUT_PATH = Path("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/data/datasets/summary_embeddings_768.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
embeddings_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved embeddings to {OUTPUT_PATH.resolve()}")


Generated embeddings for 1679 files with 768 dimensions.


,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,...,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,dim_768,file
0,0.002779,0.007182,0.016449,-0.064291,-0.019806,-0.010334,-0.013919,0.006992,0.027245,0.012633,...,-0.023677,0.014394,-0.044883,0.006649,-0.014494,-0.011969,0.008140,0.006030,0.031414,https://github.com/HasinthakaPiyumal/AI-Patter...
1,0.000060,-0.003551,0.022026,-0.067591,-0.023425,-0.003097,-0.021283,0.000361,0.013890,0.006987,...,-0.024786,0.016341,-0.044152,0.006420,-0.006976,-0.002695,0.025916,0.022266,0.039500,https://github.com/HasinthakaPiyumal/AI-Patter...
2,-0.006586,0.006543,0.022041,-0.049308,-0.025309,-0.003847,-0.001202,0.000575,0.020454,0.001272,...,-0.017744,0.009114,-0.034539,-0.002514,-0.023058,-0.003707,0.012185,0.006886,0.030473,https://github.com/HasinthakaPiyumal/AI-Patter...
3,0.003248,0.000073,-0.003037,-0.058385,-0.040864,-0.017859,-0.000622,-0.000250,0.021831,0.009276,...,-0.009886,0.016426,-0.048579,-0.016796,-0.036057,0.011685,0.023556,0.016171,0.018032,https://github.com/HasinthakaPiyumal/AI-Patter...
4,-0.010209,0.000321,0.000465,-0.061510,-0.030253,0.007674,0.008237,-0.003279,0.009458,0.003353,...,-0.019156,0.018396,-0.021749,-0.003031,-0.023755,-0.004303,0.014125,0.016087,0.009235,https://github.com/HasinthakaPiyumal/AI-Patter...


Saved embeddings to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/data/datasets/summary_embeddings_768.csv


In [ ]:
embeddings_df.shape